# PaMCP E8 — Retrieval campaign and publication figures

This notebook runs the E8 retrieval campaign from the pinned feature branch, persists datasets/rankings/results on Google Drive, exports evidence-backed E8-F1/E8-F4 CSVs, and renders the available figures.

**Important:** E8 remains `partial` while registered baselines or source conditions are blocked. B1/B1m and the distractor-count sweep are not fabricated. The notebook will show only figures with validated source data.


## Runtime

In Colab choose **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU). Run the cells in order. Public HF datasets/models do not require a token, but a Colab secret named `HF_TOKEN` can be used to reduce Hub rate-limit issues.

The first setup cell mounts Google Drive. Keep the Drive storage folder intact between sessions to preserve the pinned raw data, ToolRet snapshot, Hugging Face cache and immutable E8 run outputs.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, shutil, subprocess, sys, json, zipfile

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/msellamiTN/taas-provenance-discovery.git'
BRANCH = 'feature/pamcp-publication-visualization-v1'
WORK_DIR = Path('/content/taas-provenance-discovery')
DRIVE_ROOT = Path('/content/drive/MyDrive/PaMCP_E8')
DRIVE_DATA = DRIVE_ROOT / 'data'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
RUN_ID = 'E8-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if not (WORK_DIR / '.git').exists():
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH, REPO_URL, str(WORK_DIR)], check=True)
else:
    data_link = WORK_DIR / 'data'
    if data_link.is_symlink():
        data_link.unlink()
    subprocess.run(['git', '-C', str(WORK_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(WORK_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(WORK_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

if not DRIVE_DATA.exists():
    shutil.copytree(WORK_DIR / 'data', DRIVE_DATA)
else:
    shutil.copytree(WORK_DIR / 'data', DRIVE_DATA, dirs_exist_ok=True)
data_path = WORK_DIR / 'data'
if data_path.is_symlink():
    data_path.unlink()
elif data_path.exists():
    shutil.rmtree(data_path)
data_path.symlink_to(DRIVE_DATA, target_is_directory=True)

OUTPUT_ROOT = DRIVE_RESULTS / 'definitive' / 'E8'
VIS_DATA = DRIVE_RESULTS / 'visualization' / 'data'
VIS_FIGURES = DRIVE_RESULTS / 'visualization' / 'figures'
for path in (OUTPUT_ROOT, VIS_DATA, VIS_FIGURES):
    path.mkdir(parents=True, exist_ok=True)

os.environ['E8_RUN_ID'] = RUN_ID
os.environ['E8_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
os.environ['E8_VIS_DATA'] = str(VIS_DATA)
os.environ['E8_VIS_FIGURES'] = str(VIS_FIGURES)
os.environ['E8_DEVICE'] = 'cuda'
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'hf-cache')
os.environ['HF_HUB_CACHE'] = str(DRIVE_ROOT / 'hf-cache' / 'hub')
os.environ.setdefault('E8_TOP_K', '10')
os.environ.setdefault('E8_BOOTSTRAP_RESAMPLES', '5000')
os.environ.setdefault('E8_SEED', '20261105')

print('Branch:', BRANCH)
print('Run ID:', RUN_ID)
print('Persistent data:', DRIVE_DATA)
print('Persistent results:', DRIVE_RESULTS)


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('A CUDA GPU is required for the full E8 retrieval campaign. Choose a GPU runtime and rerun this cell.')


## Run data preparation, retrieval, export and rendering

The launcher installs the `benchmark-hf`, `visualization` and `retrieval` extras, downloads/verifies the pinned MCPToolBench++ category files, resolves immutable ToolRet and model revisions, runs the two independent retrieval tracks, then exports and renders the available figures. The full corpus and model caches are kept on Drive.


In [ ]:
subprocess.run(['bash', 'experiments/run_e8_and_plot.sh'], cwd=WORK_DIR, env=os.environ.copy(), check=True)
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_MANIFEST = json.loads((RUN_DIR / 'MANIFEST.json').read_text(encoding='utf-8'))
EXPORT_MANIFEST = json.loads((VIS_DATA / 'E8_EXPORT_MANIFEST.json').read_text(encoding='utf-8'))
print('Campaign status:', RUN_MANIFEST['status'])
print('Run directory:', RUN_DIR)
print('Figures exported:', EXPORT_MANIFEST['figures_exported'])
print('Full PaMCP single-tool adapter executed:', EXPORT_MANIFEST['primary_pa_mcp_executed'])


## Inspect and preview the results

Read the E8 manifest before interpreting any metric. The figure export manifest distinguishes registered baselines from supplementary controls and lists the figure inputs that remain unavailable.


In [ ]:
from IPython.display import display, Imageimport pandas as pd# E8-F1 is required when the exporter has at least one registered method# with valid per-task uncertainty. E8-F4 is conditional on category-level# independent-unit counts, so do not assume its CSV exists.for figure_id in ("E8-F1", "E8-F4"):    csv_path = VIS_DATA / f"{figure_id}.csv"    if csv_path.is_file():        print(f"{figure_id} input:")        display(pd.read_csv(csv_path).head(30))    else:        print(f"{figure_id}: not exported; see E8_EXPORT_MANIFEST.json")rendered_figures = sorted(VIS_FIGURES.glob("E8-F*.png"))if rendered_figures:    for figure_path in rendered_figures:        print(figure_path.name)        display(Image(filename=str(figure_path)))else:    print("No E8 figure PNGs were rendered.")print(json.dumps({    "blocked_registered_baselines": EXPORT_MANIFEST["blocked_registered_baselines"],    "other_blocked_or_not_comparable_conditions": EXPORT_MANIFEST["other_blocked_or_not_comparable_conditions"],    "figures_not_exported": EXPORT_MANIFEST["figures_not_exported"],    "bootstrap_exclusions": EXPORT_MANIFEST["bootstrap_exclusions"],}, indent=2))

In [ ]:
BUNDLE = DRIVE_ROOT / f'{RUN_ID}_publication_bundle.zip'
with zipfile.ZipFile(BUNDLE, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for base in (RUN_DIR, VIS_DATA, VIS_FIGURES):
        if base.exists():
            for file_path in base.rglob('*'):
                if file_path.is_file():
                    archive.write(file_path, file_path.relative_to(DRIVE_ROOT))
print('Publication bundle:', BUNDLE)
print('Bundle size (MiB):', round(BUNDLE.stat().st_size / (1024 ** 2), 2))

from google.colab import files
files.download(str(BUNDLE))


## E8 interpretation guardrails

- ToolRet and MCPToolBench++ results are separate benchmark tracks.
- MCPToolBench++ intervals resample target-server clusters; ToolRet intervals resample tasks.
- Only registered E8 baseline IDs with actual task-level results are in the primary figures. Extra campaign controls are not relabelled.
- `similar_tools` sensitivity scores are not B1/B1m.
- No distractor-count curve is produced until that sweep is actually run.
- Ranking metrics do not establish end-to-end execution success, safety or valid multi-tool composition.
- The campaign manifest remains authoritative; a partial run must not be described as complete.
